# Лабораторная работа №3
## Функции, типизация и исключения: шлюз биржевых заявок


### Итог работы

Вы превратите обработку заявок в набор функций с явными
контрактами. Строка внешней системы пройдёт разбор, проверку
формата, рыночные и риск-правила, расчёт комиссии и пакетную
обработку. Ошибка одной заявки не должна останавливать весь пакет.
В итоговом проекте функции будут собраны в торговую сессию, которая
контролирует деньги и позиции клиента.

### Новые инструменты

`def`, `return`, параметры, значения по умолчанию, keyword-only
параметры, аннотации типов, docstrings, локальная область видимости,
композиция функций, `raise`, `try`/`except`/`else`/`finally`,
собственные исключения и `raise ... from ...`.

Файлы, полноценные классы предметной области, декораторы, генераторы
и сторонние библиотеки пока не используются.

### Правило выполнения

Выполняйте ячейки сверху вниз и не удаляйте проверки. Функции не
должны изменять переданные словари и списки, если это прямо не
указано в контракте. Перед сдачей выполните **Restart Kernel and
Run All Cells**.


## 1. Контракт функции и собственные исключения

Контракт отвечает на четыре вопроса:

1. Какие данные принимает функция?
2. Что возвращает?
3. Какие ошибки поднимает?
4. Изменяет ли внешнее состояние?

Аннотация типа документирует намерение, но Python сам не запрещает
передать объект другого типа. Поэтому проверка внешних данных всё
равно нужна.

Небольшие классы ниже используются только как разные категории
ошибок. Полноценное проектирование классов будет в отдельной работе.


### Задание 1. Ошибки и базовые преобразования

Создайте иерархию:

- `OrderError(Exception)` — базовая ошибка шлюза;
- `OrderFormatError(OrderError)` — неверный внешний формат;
- `OrderRiskError(OrderError)` — нарушение рыночного или риск-правила;
- `OrderSettlementError(OrderError)` — недостаток денег или позиции.

Реализуйте две функции с аннотациями и docstrings:

- `normalize_symbol(raw_symbol: str) -> str`: удаляет внешние
  пробелы, переводит символ в верхний регистр; допустимы 2–8
  буквенно-цифровых знаков;
- `parse_positive_int(raw_value: str) -> int`: преобразует строку в
  строго положительное целое.

При нарушении контракта поднимайте `OrderFormatError` с сообщениями
`symbol_must_be_text`, `invalid_symbol`, `quantity_must_be_text`,
`quantity_must_be_integer`, `quantity_must_be_positive`.
Ошибку `int(...)` связывайте с новой через `raise ... from error`.


In [ ]:
class OrderError(Exception):
    pass


class OrderFormatError(Exception):
    pass


class OrderRiskError(Exception):
    pass


class OrderSettlementError(Exception):
    pass


def normalize_symbol(raw_symbol: str) -> str:
    # YOUR CODE HERE
    return ""


def parse_positive_int(raw_value: str) -> int:
    # YOUR CODE HERE
    return 0


## 2. Разбор одной заявки

Одна функция должна отвечать за одну границу системы. Парсер знает
формат строки, но не должен проверять ликвидность инструмента или
остаток денег клиента — это другие ответственности.

Строка заявки имеет шесть полей:

`order_id;symbol;side;quantity;limit_price;tier`

Пример: `O-101; SBER; BUY; 100; 255.00; BASIC`.


### Задание 2. Функция `parse_order_line`

Реализуйте `parse_order_line(line: str) -> dict[str, object]`.

Требования:

- вход должен быть строкой с шестью полями;
- `order_id` имеет вид `O-` и далее только цифры;
- symbol проверяется через `normalize_symbol`;
- side — `BUY` или `SELL`;
- quantity разбирается через `parse_positive_int`;
- limit price — положительный `float`;
- tier — `BASIC` или `PRO`;
- результат содержит ключи `order_id`, `symbol`, `side`,
  `quantity`, `limit_price`, `tier`.

Используйте сообщения `line_must_be_text`, `expected_6_fields`,
`invalid_order_id`, `invalid_side`, `limit_price_must_be_number`,
`limit_price_must_be_positive`, `invalid_tier`. Ошибку `float(...)`
связывайте через `raise ... from ...`.


In [ ]:
def parse_order_line(line: str) -> dict[str, object]:
    "Разобрать одну строку биржевой заявки."
    # YOUR CODE HERE
    return {}


## 3. Богатая сигнатура

Значения по умолчанию делают частый вызов коротким. Символ `*` в
сигнатуре запрещает случайно передавать последующие параметры по
позиции: `minimum=...` читается как часть бизнес-правила.

Функция ниже должна быть чистой: результат зависит только от
аргументов, внешние объекты не меняются.


### Задание 3. Комиссия брокера

Реализуйте:

```python
calculate_commission(
    notional: float,
    rate: float = 0.0005,
    *,
    minimum: float = 5.0,
    maximum: float | None = 100.0,
    discount: float = 0.0,
) -> float
```

Сначала рассчитайте `notional * rate * (1 - discount)`, затем
примените минимум и необязательный максимум. Верните сумму,
округлённую до двух знаков.

`notional` должен быть положительным; rate/minimum —
неотрицательными; discount принадлежит `[0, 1]`; maximum равен
`None` либо не меньше minimum. Для нарушений поднимайте `ValueError`.
Не принимайте `bool` за число.


In [ ]:
def calculate_commission(
    notional: float,
    rate: float = 0.0005,
    *,
    minimum: float = 5.0,
    maximum: float | None = 100.0,
    discount: float = 0.0,
) -> float:
    "Рассчитать комиссию с минимумом, максимумом и скидкой."
    # YOUR CODE HERE
    return 0.0


## 4. Композиция: от заявки к решению

`evaluate_order` не повторяет разбор строки: она принимает уже
нормализованный словарь и вызывает `calculate_commission`. Такая
композиция позволяет тестировать каждый уровень отдельно.

Для BUY используется цена `ask`, для SELL — `bid`. Лимит исполним,
если `ask <= limit_price` для BUY или `bid >= limit_price` для SELL.
Не достигший рынка лимит — нормальный статус `PENDING`, а не ошибка.
Нарушение риск-правила — исключение `OrderRiskError`.


In [ ]:
quotes = {
    "SBER": {"bid": 253.80, "ask": 254.20, "lot_size": 10, "daily_volume": 1_800_000},
    "GAZP": {"bid": 174.50, "ask": 174.80, "lot_size": 10, "daily_volume": 900_000},
    "YDEX": {"bid": 4_100.0, "ask": 4_110.0, "lot_size": 1, "daily_volume": 120_000},
}
quotes_snapshot = {symbol: quote.copy() for symbol, quote in quotes.items()}

batch_lines = [
    "O-101;SBER;BUY;100;255;BASIC",
    "O-102;GAZP;SELL;50;174;PRO",
    "O-103;SBER;BUY;200;250;BASIC",
    "O-104;YDEX;BUY;2;4200;PRO",
    "O-105;SBER;BUY;55;260;BASIC",
    "O-101;GAZP;BUY;10;180;BASIC",
    "broken;line",
    "O-106;GAZP;BUY;600;180;PRO",
]


### Задание 4. Функция `evaluate_order` — 3 балла

Сигнатура:

```python
evaluate_order(
    order: dict[str, object],
    quotes: dict[str, dict[str, float]],
    *,
    max_notional: float = 80_000.0,
    min_daily_volume: int = 500_000,
    fee_rate: float = 0.0005,
    fee_minimum: float = 5.0,
    fee_maximum: float | None = 100.0,
) -> dict[str, object]
```

Порядок проверок: известный symbol → кратность lot size → ликвидность
→ notional по bid/ask → max notional → достижение limit price.
Причины риск-ошибок: `unknown_symbol`,
`quantity_not_multiple_of_lot`, `daily_volume_below_minimum`,
`max_notional_exceeded`.

Результат всегда содержит `order_id`, `symbol`, `side`, `quantity`,
`tier`, `status`, `reason`, `execution_price`, `notional`,
`commission`, `cash_effect`.

Для `PENDING`: reason=`limit_not_reached`, последние четыре числовых
поля равны `None, 0.0, 0.0, 0.0`. Для APPROVED рассчитайте комиссию;
скидка PRO равна 25%. BUY даёт отрицательный cash effect, SELL —
положительный. Денежные результаты округляйте до двух знаков.


In [ ]:
def evaluate_order(
    order: dict[str, object],
    quotes: dict[str, dict[str, float]],
    *,
    max_notional: float = 80_000.0,
    min_daily_volume: int = 500_000,
    fee_rate: float = 0.0005,
    fee_minimum: float = 5.0,
    fee_maximum: float | None = 100.0,
) -> dict[str, object]:
    "Проверить заявку и рассчитать параметры возможной сделки."
    # YOUR CODE HERE
    return {}


## 5. Исключение как граница одной записи

Исключение удобно внутри конвейера, но пакетный уровень должен
решить, что с ним делать. В режиме `collect` ошибка превращается в
диагностическую запись, и обработка продолжается. В режиме `raise`
ошибка передаётся вызывающему коду немедленно.

Не используйте голый `except:`: он скроет ошибки программирования,
которые не относятся к контракту заявок.


### Задание 5. Пакетная обработка

Реализуйте:

```python
process_batch(
    lines: list[str],
    quotes: dict[str, dict[str, float]],
    *,
    on_error: str = "collect",
) -> tuple[list[dict[str, object]], list[dict[str, object]]]
```

Допустимы режимы `collect` и `raise`, иначе `ValueError` с текстом
`invalid_on_error_mode`. ID должен быть уникален во всём пакете;
повтор поднимает `OrderFormatError("duplicate_order_id")`.

В `collect` возвращайте `(results, errors)`. Ошибка содержит
`line_number` (с 1), `raw_line`, `error_type`, `message`. Ловите
только `OrderError`. ID резервируется после успешного разбора даже
тогда, когда позже заявка нарушила риск-правило.


In [ ]:
def process_batch(
    lines: list[str],
    quotes: dict[str, dict[str, float]],
    *,
    on_error: str = "collect",
) -> tuple[list[dict[str, object]], list[dict[str, object]]]:
    "Обработать пакет заявок, изолируя ожидаемые ошибки."
    # YOUR CODE HERE
    return [], []


## 6. Функция-агрегатор

Агрегатор не должен знать, как разбиралась заявка. Он получает
результаты уже установленного контракта. Это уменьшает связанность:
парсер можно изменить, не переписывая отчёт.


### Задание 6. Отчёт по пакету

Реализуйте `build_batch_report(results, errors) -> dict[str, object]`.
Верните:

- `input_count`, `approved_count`, `pending_count`, `error_count`;
- `gross_notional`, `total_commission`, `net_cash_effect` только по
  APPROVED, с округлением до двух знаков;
- `by_symbol`: для каждого исполненного symbol количество
  `approved_count` и `notional`;
- `largest_order_id`: ID исполненной заявки с максимальным notional,
  а при отсутствии — `None`.

Не изменяйте `results` и `errors`.


In [ ]:
def build_batch_report(
    results: list[dict[str, object]],
    errors: list[dict[str, object]],
) -> dict[str, object]:
    "Собрать агрегированный отчёт по результатам пакета."
    # YOUR CODE HERE
    return {}


## 7. Полная конструкция `try`

`except` выполняется при ожидаемой ошибке, `else` — только если
исключения не было, `finally` — всегда. Возврат из `finally` опасен:
он способен подавить исключение, поэтому здесь `finally` только
добавляет запись в журнал.


### Задание 7. Диагностический запуск

Реализуйте `diagnose_line(line, quotes)`. Функция возвращает
`(result, audit)`.

- в начале audit содержит `START`;
- после разбора добавьте `PARSED:<order_id>`;
- отдельно перехватите `OrderFormatError` и `OrderRiskError`,
  добавляя `ERROR:<имя класса>`; result при ошибке равен `None`;
- в `else` добавьте `RESULT:<status>`;
- в `finally` всегда добавьте `FINISH`.

Здесь ошибки сознательно превращаются в диагностический результат.
В реальном приложении выбор между записью, повторным `raise` и
восстановлением зависит от уровня программы.

Пример вывода: `["START", "PARSED:O-101", "RESULT:APPROVED", "FINISH"]`


In [ ]:
def diagnose_line(
    line: str,
    quotes: dict[str, dict[str, float]],
) -> tuple[dict[str, object] | None, list[str]]:
    "Выполнить одну заявку и вернуть подробный журнал фаз."
    # YOUR CODE HERE
    return None, []


## 8. Итоговый мини-проект

До сих пор заявки оценивались независимо. Торговая сессия добавляет
состояние клиента: доступные деньги и количество бумаг. Новая
функция должна переиспользовать парсер, evaluator и агрегатор, а не
копировать их внутреннюю логику.

### Задание 8. Торговая сессия клиента

Реализуйте:

```python
run_trading_session(
    lines: list[str],
    quotes: dict[str, dict[str, float]],
    *,
    starting_cash: float,
    starting_positions: dict[str, int],
) -> dict[str, object]
```

Для каждой строки:

1. разберите заявку и проверьте уникальность ID;
2. получите результат `evaluate_order`;
3. PENDING сохраните без изменения состояния;
4. перед BUY проверьте деньги, иначе поднимите
   `OrderSettlementError("insufficient_cash")`;
5. перед SELL проверьте позицию, иначе
   `OrderSettlementError("insufficient_position")`;
6. исполненную сделку примените к cash и positions;
7. перехватите только `OrderError` и добавьте ошибку с ключами
   `line_number`, `order_id`, `error_type`, `message`.

ID резервируется после разбора. Возвращаемый словарь содержит
`cash` (два знака), `positions`, `results`, `errors`, `report` от
`build_batch_report`. Исходные позиции и строки не изменяйте.


In [ ]:
session_lines = [
    "O-201;SBER;BUY;100;255;PRO",
    "O-202;GAZP;SELL;50;174;BASIC",
    "O-203;SBER;BUY;300;250;BASIC",
    "O-204;YDEX;BUY;2;4200;PRO",
    "O-205;SBER;BUY;400;260;BASIC",
    "O-206;GAZP;SELL;100;170;PRO",
    "O-207;SBER;BUY;200;260;PRO",
]
starting_positions = {"SBER": 0, "GAZP": 100, "YDEX": 0}
positions_snapshot = starting_positions.copy()
lines_snapshot = session_lines.copy()


In [ ]:
def run_trading_session(
    lines: list[str],
    quotes: dict[str, dict[str, float]],
    *,
    starting_cash: float,
    starting_positions: dict[str, int],
) -> dict[str, object]:
    "Обработать заявки с учётом денег и позиций клиента."
    # YOUR CODE HERE
    return {}


session_result = run_trading_session(
    session_lines,
    quotes,
    starting_cash=60_000.0,
    starting_positions=starting_positions,
)


## Вывод и защита

Добавьте после этой ячейки собственный вывод на 8–12 предложений:

1. Какие ответственности разделены между parser, evaluator, batch
   processor и trading session?
2. Почему PENDING не является исключением?
3. Почему риск-ошибка и settlement-ошибка имеют разные типы?
4. Где использована композиция функций?
5. Почему O-206 не изменила cash и позицию?
6. Как доказать, что входные словари не изменены?
7. Какие два граничных случая вы бы добавили в тесты?

### Контрольные вопросы

1. Что входит в контракт функции?
2. Проверяет ли Python аннотации типов автоматически?
3. Чем `return` отличается от `print`?
4. Для чего нужны keyword-only параметры?
5. Когда следует поднимать исключение, а когда возвращать статус?
6. Зачем создавать собственную иерархию исключений?
7. Что делает `raise ... from error`?
8. В каком случае выполняется блок `else` конструкции `try`?
9. Почему не следует возвращать значение из `finally`?
10. Что такое чистая функция?
11. Почему голый `except:` опасен?
12. Что означает «ошибка одной записи изолирована»?


_Напишите здесь собственный вывод._
